In [1]:
import requests as req

### 네이버 페이지 요청하기

In [2]:
res = req.get("http://www.naver.com")

### 응답 확인

In [3]:
res

<Response [200]>

In [4]:
res.text

'   <!doctype html> <html lang="ko" class="fzoom"> <head> <meta charset="utf-8"> <meta name="Referrer" content="origin"> <meta http-equiv="X-UA-Compatible" content="IE=edge"> <meta name="viewport" content="width=1190"> <title>NAVER</title> <meta name="apple-mobile-web-app-title" content="NAVER"/> <meta name="robots" content="index,nofollow"/> <meta name="description" content="네이버 메인에서 다양한 정보와 유용한 컨텐츠를 만나 보세요"/> <meta property="og:title" content="네이버"> <meta property="og:url" content="https://www.naver.com/"> <meta property="og:image" content="https://s.pstatic.net/static/www/mobile/edit/2016/0705/mobile_212852414260.png"> <meta property="og:description" content="네이버 메인에서 다양한 정보와 유용한 컨텐츠를 만나 보세요"/> <meta name="twitter:card" content="summary"> <meta name="twitter:title" content=""> <meta name="twitter:url" content="https://www.naver.com/"> <meta name="twitter:image" content="https://s.pstatic.net/static/www/mobile/edit/2016/0705/mobile_212852414260.png"> <meta name="twitter:description" 

### 해당 페이지의 title만 출력해보기

In [5]:
import requests

url = 'https://snuco.snu.ac.kr/foodmenu/'
html = requests.get(url).text

title_begin = html.index('<title>')
title_end = html.index('</title>')
title = html[title_begin+len('<title>') : title_end]

print(title)

식단 - 서울대학교 생활협동조합


### 주식의 종목코드로 기업 이름 알아내기

In [6]:
import requests

code = '005930'
url = 'https://finance.naver.com/item/main.nhn?code='
html = requests.get(url + code).text

title_begin = html.index('<title>')
title_end = html.index('</title>')
title = html[title_begin : title_end]

print(title)

<title>삼성전자 : Npay 증권


# OPEN API 사용

- https://www.data.go.kr/data/15098771/openapi.do

In [8]:
!pip install python-dotenv


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import requests
import os
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경변수에서 서비스키 불러오기
key = os.getenv("API_KEY")

url = 'http://apis.data.go.kr/1352000/ODMS_COVID_05/callCovid05Api' \
      + '?serviceKey=' + key \
      + '&pageNo=1' \
      + '&numOfRows=10' \
      + '&create_dt=2022-01-08'

xml = requests.get(url).text
print(xml)


<?xml version="1.0" encoding="UTF-8"?>
<response>
 <header>
  <resultCode>00</resultCode>
  <resultMsg>NORMAL SERVICE</resultMsg>
 </header>
 <body>
  <items>
   <item>
    <confCase>53179</confCase>
    <confCaseRate>8.05</confCaseRate>
    <criticalRate>0.01</criticalRate>
    <death>3</death>
    <deathRate>0.05</deathRate>
    <gubun>0-9</gubun>
    <createDt>2022-01-08</createDt>
   </item>
   <item>
    <confCase>98569</confCase>
    <confCaseRate>14.91</confCaseRate>
    <criticalRate>0.01</criticalRate>
    <death>13</death>
    <deathRate>0.22</deathRate>
    <gubun>20-29</gubun>
    <createDt>2022-01-08</createDt>
   </item>
   <item>
    <confCase>95445</confCase>
    <confCaseRate>14.44</confCaseRate>
    <criticalRate>0.3</criticalRate>
    <death>287</death>
    <deathRate>4.79</deathRate>
    <gubun>50-59</gubun>
    <createDt>2022-01-08</createDt>
   </item>
   <item>
    <confCase>39526</confCase>
    <confCaseRate>5.98</confCaseRate>
    <criticalRate>4.16</criticalRa

In [11]:
from bs4 import BeautifulSoup

xml = requests.get(url).text
bs = BeautifulSoup(xml, "xml") 
items = bs.select('item')

for item in items:
    confCase = item.select_one('confCase').text
    gubun = item.select_one('gubun').text
    print('구분', gubun, '확진자', confCase)


FeatureNotFound: Couldn't find a tree builder with the features you requested: xml. Do you need to install a parser library?

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("API_KEY")

base_url = "http://apis.data.go.kr/1352000/ODMS_COVID_05/callCovid05Api"

# 날짜 범위 설정
start_date = datetime(2022, 1, 1)
end_date = datetime(2022, 1, 10)

date_list = [
    (start_date + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range((end_date - start_date).days + 1)
]


In [ ]:
data = []

for date in date_list:
    url = (
        f"{base_url}"
        f"?serviceKey={key}"
        f"&pageNo=1"
        f"&numOfRows=10"
        f"&create_dt={date}"
    )

    response = requests.get(url)
    bs = BeautifulSoup(response.text, "xml")
    items = bs.select("item")

    for item in items:
        gubun = item.select_one("gubun").text
        confCase = item.select_one("confCase").text

        data.append({
            "날짜": date,
            "구분": gubun,
            "확진자수": int(confCase)
        })


In [ ]:
df = pd.DataFrame(data)
df.tail()


,날짜,구분,확진자수
95,2022-01-10,50-59,96230
96,2022-01-10,10-19,67513
97,2022-01-10,남성,346703
98,2022-01-10,20-29,99575
99,2022-01-10,0-9,53998


In [ ]:
# 결측/중복 점검
print(df.isna().sum())
print("중복 행 개수:", df.duplicated().sum())

df.head()


날짜      0
구분      0
확진자수    0
dtype: int64
중복 행 개수: 0


,날짜,구분,확진자수
0,2022-01-01,20-29,94950
1,2022-01-01,50-59,92150
2,2022-01-01,30-39,92347
3,2022-01-01,60-69,90422
4,2022-01-01,70-79,38608


In [ ]:
# 연령대 데이터만 남기기 (숫자-숫자 or '이상' 포함)
age_df = df[df['구분'].str.contains(r'\d')].copy()


In [ ]:
def extract_age(group):
    if '이상' in group:
        return int(group.replace(' 이상', ''))
    else:
        return int(group.split('-')[0])

age_df['연령대'] = age_df['구분'].apply(extract_age)


In [ ]:
pivot_df = pd.pivot_table(
    age_df,
    values='확진자수',
    index='날짜',
    columns='연령대',
    aggfunc='sum'
)

pivot_df


연령대,0,10,20,30,40,50,60,70,80
날짜,,,,,,,,,
2022-01-01,49825.0,NaN,94950.0,92347.0,92810.0,92150.0,90422.0,38608.0,20618.0
2022-01-02,50345.0,63989.0,95429.0,92929.0,93405.0,92648.0,90880.0,38763.0,20695.0
2022-01-03,50834.0,64376.0,95818.0,93403.0,93897.0,93010.0,91227.0,38886.0,20756.0
2022-01-04,51249.0,64732.0,96241.0,93877.0,94355.0,93403.0,NaN,38984.0,20821.0
2022-01-05,51771.0,65287.0,96819.0,94631.0,95112.0,93963.0,92004.0,39153.0,20929.0
2022-01-06,NaN,65794.0,97431.0,95262.0,95799.0,94469.0,92411.0,39283.0,21018.0
2022-01-07,52769.0,66248.0,98000.0,95875.0,96403.0,NaN,92763.0,39394.0,21105.0
2022-01-08,53179.0,NaN,98569.0,96428.0,96928.0,95445.0,93129.0,39526.0,21156.0
2022-01-09,53602.0,67110.0,99098.0,96958.0,NaN,95860.0,93452.0,39636.0,21222.0


## naver api 뉴스 검색

- https://developers.naver.com/apps/#/register
- 해당 사이트에서 api 키를 발급받습니다.

In [12]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# 환경변수 불러오기
CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

url = "https://openapi.naver.com/v1/search/news.json"

headers = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET
}

params = {
    "query": "인공지능",
    "display": 5,
    "start": 1,
}

response = requests.get(url, headers=headers, params=params)
data = response.json()

# 결과 확인
print(data)


{'lastBuildDate': 'Mon, 12 Jan 2026 15:30:11 +0900', 'total': 3585178, 'start': 1, 'display': 5, 'items': [{'title': '부산 기장군, 군민 정보화교육에 <b>인공지능</b>(AI) 교육과정 도입', 'originallink': 'http://www.lawissue.co.kr/view.php?ud=2026011215235459719a8c8bf58f_12', 'link': 'http://www.lawissue.co.kr/view.php?ud=2026011215235459719a8c8bf58f_12', 'description': '부산 기장군(군수 정종복)은 1월 19일부터 급속히 확산되는 <b>인공지능</b>(AI) 기술변화에 선제적으로 대응하고 군민의 디지털 활용 역량을 높이기 위해, 군민 정보화교육에 <b>인공지능</b>(AI) 과정을 도입하는 특수시책을... ', 'pubDate': 'Mon, 12 Jan 2026 15:28:00 +0900'}, {'title': "서울시, '창의행정'으로 글로벌 톱5·강북 전성시대 '본격' 시동", 'originallink': 'https://www.asiatoday.co.kr/view.php?key=20260112010005293', 'link': 'https://www.asiatoday.co.kr/view.php?key=20260112010005293', 'description': '특히 6·3 지방선거를 4개월여 앞둔 시점에서 행정 전반에 <b>인공지능</b>(AI) 활용을 확대하고, 강북 대개조를 중심으로 도시 구조를 바꿔 시민이 체감하는 변화를 끌어올려 글로벌 톱5 도시로 업그레이드하겠다는 구상이다.... ', 'pubDate': 'Mon, 12 Jan 2026 15:28:00 +0900'}, {'title': '기상청, 제43회 기상기후 사진·콘텐츠 공모전 개최', 'originallink': 'https://www.pub

In [13]:
items = data.get("items", [])
print("items count:", len(items))

# 첫 데이터만 구조 확인
if items:
    print(items[0].keys())


items count: 5
dict_keys(['title', 'originallink', 'link', 'description', 'pubDate'])


In [14]:
import pandas as pd
import re
from html import unescape

def clean_html(text: str) -> str:
    if text is None:
        return ""
    text = unescape(text)
    # <b>...</b> 같은 태그 제거
    return re.sub(r"<.*?>", "", text)

rows = []
for it in items:
    rows.append({
        "title": clean_html(it.get("title")),
        "description": clean_html(it.get("description")),
        "link": it.get("link"),
        "pubDate": it.get("pubDate"),
        "originallink": it.get("originallink"),
    })

df = pd.DataFrame(rows)
df

,title,description,link,pubDate,originallink
0,"부산 기장군, 군민 정보화교육에 인공지능(AI) 교육과정 도입",부산 기장군(군수 정종복)은 1월 19일부터 급속히 확산되는 인공지능(AI) 기술변...,http://www.lawissue.co.kr/view.php?ud=20260112...,"Mon, 12 Jan 2026 15:28:00 +0900",http://www.lawissue.co.kr/view.php?ud=20260112...
1,"서울시, '창의행정'으로 글로벌 톱5·강북 전성시대 '본격' 시동",특히 6·3 지방선거를 4개월여 앞둔 시점에서 행정 전반에 인공지능(AI) 활용을 ...,https://www.asiatoday.co.kr/view.php?key=20260...,"Mon, 12 Jan 2026 15:28:00 +0900",https://www.asiatoday.co.kr/view.php?key=20260...
2,"기상청, 제43회 기상기후 사진·콘텐츠 공모전 개최","공모 부문은 사진, 영상, 생성형 인공지능(AI) 등 3가지 분야로 나뉜다. 특히 ...",https://www.public25.com/news/articleView.html...,"Mon, 12 Jan 2026 15:28:00 +0900",https://www.public25.com/news/articleView.html...
3,AI 전문인력 양성 위한 ‘AI 캠퍼스’ 신설…훈련수당도 지급,정부가 인공지능(AI) 전문인력 1만 명 양성을 지원한다. 고용노동부와 한국기술교육...,https://n.news.naver.com/mnews/article/020/000...,"Mon, 12 Jan 2026 15:28:00 +0900",https://www.donga.com/news/Society/article/all...
4,"카카오 준법과신뢰위원회, '연간보고서 2025' 발간",정 의장은 준신위가 형성한 준법 신뢰 중심 문화를 기반으로 인공지능(AI) 생태계 ...,https://www.g-enews.com/view.php?ud=2026011214...,"Mon, 12 Jan 2026 15:28:00 +0900",https://www.g-enews.com/view.php?ud=2026011214...


In [15]:
df["pubDate"] = pd.to_datetime(df["pubDate"], errors="coerce")
df = df.sort_values("pubDate", ascending=False).reset_index(drop=True)

df[["pubDate", "title"]].head()


,pubDate,title
0,2026-01-12 15:28:00+09:00,"부산 기장군, 군민 정보화교육에 인공지능(AI) 교육과정 도입"
1,2026-01-12 15:28:00+09:00,"서울시, '창의행정'으로 글로벌 톱5·강북 전성시대 '본격' 시동"
2,2026-01-12 15:28:00+09:00,"기상청, 제43회 기상기후 사진·콘텐츠 공모전 개최"
3,2026-01-12 15:28:00+09:00,AI 전문인력 양성 위한 ‘AI 캠퍼스’ 신설…훈련수당도 지급
4,2026-01-12 15:28:00+09:00,"카카오 준법과신뢰위원회, '연간보고서 2025' 발간"


In [16]:
output_path = "naver_news_ai.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("saved:", output_path)


saved: naver_news_ai.csv
